## Inverse Kinemaics - Jacobian based methods 
- Test IK methods based on Jacobian
    - IK can be solved by multiplying Inverse Matrix of Jacobian
    - Several methods to solve inverse jacobian without singularity

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [2]:
model_path = "../ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

Get Mujoco Jacobian

In [3]:
def get_jac_body_name(body_name=None):

    # initialize positional & rotational jacobian
    Jacobian_p = np.zeros((3,model.nu))
    Jacobian_r = np.zeros((3,model.nu))

    # get jacobian of end-effector
    mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
    return Jacobian_p, Jacobian_r

### Function: calculate inverse jacobian

In [4]:
def get_invjac_name(body_name, method='svd', sigma_threshold=0.2, damping=1.0):

    jacobian_p, jacobian_r = get_jac_body_name(body_name)

    if method=='svd':
        # get inverse jacobian with Singular Value Decomposition
        U, Sigma, V = np.linalg.svd(jacobian_p, compute_uv=True)

        # past implementation - not good
        # Sigma_clipped_rev = np.minimum(1 / Sigma, upper_bound)

        # suppress singularities modifying sigma
        Sigma_clipped_rev = np.zeros_like(Sigma)
        for i, value in enumerate(Sigma):
            if Sigma[i] < sigma_threshold:
                Sigma_clipped_rev[i] = 0
            else:
                Sigma_clipped_rev[i] = 1/Sigma[i]

        # inverse matrix for position jacobian
        S_rev_matrix = np.zeros((model.nu,3)) # positional dimension = 3, dof = model.nu
        for i, value in enumerate(Sigma_clipped_rev):
            S_rev_matrix[i,i] = value
        J_inverse = V @ S_rev_matrix @ U.T

    if method=='DLS':
        # apply damped least squares
        pass
    
    return J_inverse

### MAIN loop: calculate error & update with forward

In [9]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# initialize robot
init_qpos = [3.14, -0.8, -2.5, -2.2, 0.0, 0.0]

mujoco.mj_resetData(model, data)
data.qpos = init_qpos
mujoco.mj_forward(model, data) # first forward to get jacobian with no error

# goal position
goal_pos = [0.4, 0.4, 0.2]
body_name = "wrist_3_link"


while True:
    if viewer.is_alive:

        # get inverse jacobian & unit error vector
        J_inverse = get_invjac_name(body_name=body_name, method='svd')
        error = goal_pos - data.body(body_name).xpos.copy()
        print(f"current error: {error}")
        error /= np.linalg.norm(error)

        dq = J_inverse @ error
        dq = np.clip(dq, -0.01, 0.01)
        print(f"dq: {dq}")
        data.qpos -= dq
        # print(f"qpos before update: {qpos_before} \n qpos after update: {data.qpos}")

        mujoco.mj_forward(model, data)

        print(f"current body pos: {data.body(body_name).xpos}")

        # terminalize
        if np.linalg.norm(goal_pos - data.body(body_name).xpos) < 0.02:
            print("IK done.")
            break

        viewer.render()

    else:
        break

# close
viewer.close()

current error: [ 0.26574288  0.23866787 -0.13517305]
dq: [ 1.00000000e-02 -1.00000000e-02  1.00000000e-02  1.00000000e-02
  7.58996119e-17  0.00000000e+00]
current body pos: [0.13582027 0.15623567 0.33149487]
current error: [ 0.26417973  0.24376433 -0.13149487]
dq: [ 1.00000000e-02 -1.00000000e-02  1.00000000e-02  1.00000000e-02
 -4.18326485e-17  0.00000000e+00]
current body pos: [0.13729542 0.15114739 0.32779365]
current error: [ 0.26270458  0.24885261 -0.12779365]
dq: [ 1.00000000e-02 -1.00000000e-02  1.00000000e-02  1.00000000e-02
 -1.60739219e-16  0.00000000e+00]
current body pos: [0.13868314 0.14606888 0.32406977]
current error: [ 0.26131686  0.25393112 -0.12406977]
dq: [ 1.0000000e-02 -1.0000000e-02  1.0000000e-02  1.0000000e-02
 -2.3227916e-16  0.0000000e+00]
current body pos: [0.13998401 0.14100176 0.3203236 ]
current error: [ 0.26001599  0.25899824 -0.1203236 ]
dq: [ 1.00000000e-02 -1.00000000e-02  1.00000000e-02  1.00000000e-02
 -8.00908618e-17  0.00000000e+00]
current body p